In [0]:
!pip install plotly
# Import Libraries
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pyspark.sql import functions as F

# Read Data from Gold Layer
df_daily = spark.table("us_economy_db.gold_market_sentiment_summary").toPandas()
df_monthly = spark.table("us_economy_db.gold_monthly_trends").toPandas()
df_authors = spark.table("us_economy_db.gold_top_market_voices").toPandas()

print(f"Successfully loaded {len(df_daily)} daily/hourly records and {len(df_monthly)} monthly records.")

# Executive KPIs Summary

In [0]:
total_posts = df_daily["total_posts"].sum()
total_bullish = df_daily["bullish_positive_posts"].sum()
total_bearish = df_daily["bearish_negative_posts"].sum()
overall_sentiment = df_daily["avg_sentiment_score"].mean()

bullish_pct = round((total_bullish / total_posts) * 100, 1)
bearish_pct = round((total_bearish / total_posts) * 100, 1)

print("=" * 45)
print(" Market Indicators Summary (US Economy & Gold 2026)")
print("=" * 45)
print(f"• Total economic posts analyzed: {total_posts:,}")
print(f"• Overall average sentiment score: {overall_sentiment:.3f}")
print(f"• Bullish percentage: {bullish_pct}% ({total_bullish:,} posts)")
print(f"• Bearish percentage: {bearish_pct}% ({total_bearish:,} posts)")
print("=" * 45)

# Compilation of monthly average sentiment for each economic topic

In [0]:
monthly_topic_trend = (
df_monthly.groupby(["year_month", "search_topic"], as_index=False)
.agg(
avg_score=("avg_sentiment_score", "mean"),
total_volume=("post_count", "sum"),
)
.sort_values("year_month")
)

fig_trend = px.line(
monthly_topic_trend,
x="year_month",
y="avg_score",
color="search_topic",
markers=True,
title="Evolution of Market Sentiment Toward Gold and the U.S. Economy During 2026",
labels={
"year_month": "Month",
"avg_score": "Average Sentiment Score",
"search_topic": "Economic Topic",
},
template="plotly_dark",
)

fig_trend.add_hline(y=0, line_dash="dash", line_color="white", annotation_text="Neutral Line")
fig_trend.show()

# Comparison of Bullish vs. Bearish sentiment for each topic (Stacked Bar Chart)

In [0]:
topic_sentiment_dist = (
df_monthly.groupby(["search_topic", "sentiment_label"], as_index=False)["post_count"].sum()
)

fig_bar = px.bar(
topic_sentiment_dist,
x="search_topic",
y="post_count",
color="sentiment_label",
title="Sentiment Distribution (Positive / Negative / Neutral) by Economic Topic",
labels={
"search_topic": "Topic",
"post_count": "Number of Posts",
"sentiment_label": "Classification",
},
color_discrete_map={
"Positive": "#00CC96",  # Green for optimism
"Negative": "#EF553B",  # Red for pessimism
"Neutral": "#636EFA",   # Blue for neutrality
},
barmode="group",
template="plotly_dark",
)

fig_bar.show()

# Analysis of Peak Times and Daily Activity (Heatmap)

In [0]:
hourly_activity = (
df_daily.groupby(["search_topic", "post_hour"], as_index=False)["avg_sentiment_score"].mean()
)

heatmap_data = hourly_activity.pivot(
index="search_topic", columns="post_hour", values="avg_sentiment_score"
)

fig_heatmap = px.imshow(
heatmap_data,
aspect="auto",
color_continuous_scale="RdYlGn",  # From red (negative) to green (positive)
title="Sentiment Heatmap by Hour of the Day (UTC) and Economic Topic",
labels=dict(x="Hour (UTC)", y="Economic Topic", color="Average Sentiment"),
template="plotly_dark",
)

fig_heatmap.show()

# Top 15 Influential Accounts in Discussions on Gold and the Economy (Horizontal Bar Chart)

In [0]:
top_15_authors = (
df_authors.groupby("author_handle", as_index=False)
.agg(total_posts=("total_posts", "sum"), avg_sentiment=("avg_sentiment", "mean"))
.sort_values("total_posts", ascending=True)
.tail(15)
)

fig_authors = px.bar(
top_15_authors,
x="total_posts",
y="author_handle",
orientation="h",
color="avg_sentiment",
color_continuous_scale="RdYlGn",
title="Top 15 Most Active and Influential Accounts in Gold and Economic Discussions",
labels={
"total_posts": "Total Posts",
"author_handle": "Account",
"avg_sentiment": "Average Sentiment",
},
template="plotly_dark",
)

fig_authors.show()